[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C35_Speech_Audio_Course/05_speech_llms/05_speech_llms.ipynb)

# 05 · 语音 LLM 与流式（用 numpy 从零实现）

目标：把 **音频 token 化、玩具离散语音 LM（n-gram）、流式分块** 用 numpy 从零实现，
并对拍核心不变量——**语音 LM 困惑度 < 词表大小（均匀基线）**、**流式分块预测 == 整段预测**。

路线：音频 token 化 → n-gram 计数 LM → 概率归一 → 困惑度<K(对拍) → 自回归采样 → 流式分块(对拍) → ✏️ 练习 → 📖 答案 → 🧪 AudioLM/Moshi 胶囊。

> 心智模型：**音频 token 序列和文本 token 序列没有本质区别**；文本 LM 的一切(n-gram/困惑度/采样)原样适用。

## 1 · 音频 token 化：波形 → 整数序列

承接模块 03：把波形分帧、提特征、VQ 量化成整数 token 序列。
这就是语音 LM 的『词表』。对拍：所有 token 落在 `[0, K)`。

In [ ]:
import numpy as np
from collections import defaultdict
rng = np.random.default_rng(0)

def hann_window(N):
    return 0.5 - 0.5*np.cos(2*np.pi*np.arange(N)/N)

def vq_encode(x, codebook):
    d2 = ((x[:,None,:] - codebook[None,:,:])**2).sum(-1)
    return np.argmin(d2, axis=1)

def tokenize_audio(x, codebook, frame_len=256, hop=128):
    '''波形 -> 帧的幅度谱特征 -> VQ -> token 序列。'''
    T = 1 + (len(x) - frame_len)//hop
    idx = np.arange(frame_len)[None,:] + hop*np.arange(T)[:,None]
    feats = np.abs(np.fft.rfft(x[idx]*hann_window(frame_len)[None,:], axis=1))
    return vq_encode(feats, codebook)

K = 16                              # 词表大小（码本大小）
F = 256//2 + 1
codebook = rng.standard_normal((K, F))
# 一段结构化波形（变频啁啾，token 会有规律）
t = np.arange(8000)/1000.0
x = np.sin(2*np.pi*(5 + 3*t)*t)
tokens = tokenize_audio(x, codebook)
print('token 序列长度', len(tokens), '| 前 20 个:', tokens[:20])
assert tokens.min() >= 0 and tokens.max() < K
print(f'✅ 音频 -> {len(tokens)} 个整数 token（词表 K={K}）—— 这就是语音 LM 的输入')

## 2 · n-gram 语音 LM：计数估计条件概率

用前 `n-1` 个 token 的频次估计下一个 token：`p(u_t | u_{t-n+1..t-1})`。
加 1 平滑（Laplace）避免零概率。对拍：每个上下文的条件分布**归一**（和为 1）。

In [ ]:
def train_ngram(tokens, n=2, K=16):
    '''返回 (上下文 -> 计数向量) 字典。'''
    counts = defaultdict(lambda: np.zeros(K))
    for i in range(len(tokens) - n + 1):
        ctx = tuple(tokens[i:i+n-1])
        nxt = tokens[i+n-1]
        counts[ctx][nxt] += 1
    return counts

def ngram_prob(counts, ctx, K=16, alpha=1.0):
    '''加 alpha 平滑的条件概率分布 p(·|ctx)。'''
    c = counts.get(ctx, np.zeros(K))
    return (c + alpha) / (c.sum() + alpha*K)

counts = train_ngram(tokens, n=2, K=K)
# 对拍：每个上下文的条件分布归一
for ctx in list(counts.keys())[:5]:
    pdist = ngram_prob(counts, ctx, K)
    assert abs(pdist.sum() - 1.0) < 1e-9, ctx
# 未见过的上下文 -> 均匀（平滑兜底）
unseen = ngram_prob(counts, (999,), K)
assert np.allclose(unseen, 1.0/K)
print(f'学到 {len(counts)} 个上下文的条件分布，均归一')
print('✅ n-gram 语音 LM：计数 + 平滑，条件概率合法')

## 3 · 困惑度 < 词表大小：证明学到了结构 ★

困惑度 `PPL = exp(平均负对数似然)`。**均匀基线 = K**（瞎猜）。
音频 token 远非均匀（相邻帧相关、有重复），所以学过的 LM 的 PPL 应**显著 < K**。

In [ ]:
def perplexity(counts, tokens, n=2, K=16, alpha=1.0):
    '''在 tokens 上评估 n-gram LM 的困惑度。'''
    nll = 0.0; cnt = 0
    for i in range(len(tokens) - n + 1):
        ctx = tuple(tokens[i:i+n-1])
        nxt = tokens[i+n-1]
        pdist = ngram_prob(counts, ctx, K, alpha)
        nll += -np.log(pdist[nxt])
        cnt += 1
    return float(np.exp(nll / cnt))

# 训练/评估分开（用同源但不同段的 token；这里简化用前后段）
split = len(tokens)*2//3
train_tok, eval_tok = tokens[:split], tokens[split:]
counts = train_ngram(train_tok, n=2, K=K)
ppl = perplexity(counts, eval_tok, n=2, K=K)
uniform_ppl = K
print(f'2-gram 语音 LM 困惑度 = {ppl:.2f}')
print(f'均匀基线困惑度       = {uniform_ppl} (= 词表大小 K)')
assert ppl < uniform_ppl, 'LM 应优于均匀瞎猜'
# 训练集上困惑度更低（学到了）
ppl_train = perplexity(counts, train_tok, n=2, K=K)
assert ppl_train < uniform_ppl
print(f'✅ ★ 困惑度 {ppl:.2f} < K={K} —— 语音 LM 确实捕捉到了音频 token 的结构')

## 4 · 更长上下文 → 更低困惑度

更大的 `n`（更长历史）应在训练分布上给出更低困惑度（捕捉更长程依赖）。
用一个有较强重复结构的 token 序列验证 n=1,2,3 的困惑度递减。

In [ ]:
# 造一个有强结构(周期重复)的 token 序列
pattern = [3, 7, 7, 1, 9, 9, 2]
struct_tokens = np.array(pattern * 40)
print('结构化序列(周期=7):', struct_tokens[:14], '...')
ppls = {}
for n in [1, 2, 3]:
    cnt = train_ngram(struct_tokens, n=n, K=K)
    ppls[n] = perplexity(cnt, struct_tokens, n=n, K=K)
    print(f'n={n}: 困惑度 = {ppls[n]:.3f}')
# 更长上下文捕捉周期 -> 困惑度递减
assert ppls[3] < ppls[2] < ppls[1], '更长上下文应更低困惑度'
assert ppls[1] < K
print('✅ 更长上下文(更大 n) -> 更低困惑度 —— 捕捉更长程的 token 依赖')

## 5 · 自回归采样：用语音 LM『续写』token

和文本 LM 一样：给个起始上下文，反复按 `p(下一个|历史)` 采样，生成新 token 序列。
（生成的 token 再经模块 04 解码器就能变回波形——这就是『生成音频』。）

In [ ]:
def sample_ngram(counts, prompt, length, n=2, K=16, alpha=1.0, seed=0):
    g = np.random.default_rng(seed)
    seq = list(prompt)
    for _ in range(length):
        ctx = tuple(seq[-(n-1):]) if n > 1 else ()
        pdist = ngram_prob(counts, ctx, K, alpha)
        seq.append(int(g.choice(K, p=pdist)))
    return seq

cnt = train_ngram(struct_tokens, n=3, K=K)
gen = sample_ngram(cnt, prompt=[3, 7], length=20, n=3, K=K, alpha=0.01)
print('起始 [3,7] -> 生成:', gen)
# 生成的 token 都合法
assert all(0 <= u < K for u in gen)
# 低平滑 + 强结构 -> 生成应大致复现周期模式(7 后常跟 7)
cnt7 = train_ngram(struct_tokens, n=2, K=K)
after7 = ngram_prob(cnt7, (7,), K, alpha=0.01)
assert after7.argmax() in (7, 1), '7 之后最可能是 7 或 1(数据里的模式)'
print('✅ 自回归采样生成合法 token，复现了训练数据的结构（7→7→1...）')

## 6 · 流式分块 == 整段处理 ★

流式：把序列分块、逐块预测、维护跨块历史。
对 n-gram（只依赖前 n-1 个 token），**只要每块保留前 n-1 个边界 token，分块结果与整段逐位相等**——流式无损。

In [ ]:
def predict_all(counts, tokens, n=2, K=16, alpha=1.0):
    '''整段：对每个位置给出 p(u_t|历史) 的预测分布，返回 (位置, K) 矩阵。'''
    out = []
    for i in range(n-1, len(tokens)):
        ctx = tuple(tokens[i-(n-1):i])
        out.append(ngram_prob(counts, ctx, K, alpha))
    return np.array(out)

def predict_streaming(counts, tokens, chunk=10, n=2, K=16, alpha=1.0):
    '''流式：分块处理，每块带前 n-1 个 token 的前文（lookback）。'''
    out = []
    pos = n-1
    while pos < len(tokens):
        end = min(pos + chunk, len(tokens))
        for i in range(pos, end):
            ctx = tuple(tokens[i-(n-1):i])   # 仅需前 n-1 个 -> 跨块只靠边界
            out.append(ngram_prob(counts, ctx, K, alpha))
        pos = end
    return np.array(out)

counts = train_ngram(tokens, n=3, K=K)
full = predict_all(counts, tokens, n=3, K=K)
strm = predict_streaming(counts, tokens, chunk=7, n=3, K=K)
print('整段预测形状', full.shape, '| 流式预测形状', strm.shape)
assert full.shape == strm.shape
assert np.allclose(full, strm), '流式与整段必须逐位相等'
print('✅ ★ 流式分块预测 == 整段预测（逐位相等）—— n-gram 流式无损')

---
## ✏️ 练习 1：音频 token 化

实现 `audio_to_tokens(x, codebook, N, H)`：分帧、取 rfft 幅度、VQ。返回整数 token 序列。

In [ ]:
def audio_to_tokens(x, codebook, N=256, H=128):
    # TODO: T=1+(len(x)-N)//H; 广播下标切帧; feats=|rfft(帧*hann)|; 返回 vq_encode(feats, codebook)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
cb = rng.standard_normal((8, 256//2+1))
xx = np.sin(2*np.pi*3*np.arange(4000)/1000.0)
toks = audio_to_tokens(xx, cb)
assert toks.min() >= 0 and toks.max() < 8
assert len(toks) == 1 + (4000-256)//128
print(f'✅ 练习 1 通过：音频 -> {len(toks)} 个 token，均在 [0,8)')

## ✏️ 练习 2：n-gram 计数与条件概率

实现 `count_ngrams(tokens, n, K)`（返回 上下文->计数向量 dict）与 `cond_prob(counts, ctx, K, alpha)`（加平滑、归一）。

In [ ]:
def count_ngrams(tokens, n, K):
    # TODO: defaultdict(lambda: zeros(K)); 遍历, ctx=前n-1, nxt=第n; counts[ctx][nxt]+=1
    raise NotImplementedError

def cond_prob(counts, ctx, K, alpha=1.0):
    # TODO: c=counts.get(ctx, zeros(K)); 返回 (c+alpha)/(c.sum()+alpha*K)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
toks = np.array([0,1,1,0,1,1,0,1,1])
cnt = count_ngrams(toks, n=2, K=4)
# 上下文(1,)后面: 出现 1,0,1,0,1 -> 主要是 0 和 1
p1 = cond_prob(cnt, (1,), 4)
assert abs(p1.sum() - 1.0) < 1e-9
assert p1.argmax() in (0, 1)
# 未见上下文 -> 均匀
assert np.allclose(cond_prob(cnt, (3,), 4), 0.25)
print('✅ 练习 2 通过：n-gram 计数与条件概率正确')

## ✏️ 练习 3：困惑度

实现 `compute_perplexity(counts, tokens, n, K, alpha)`：`exp(平均 -log p(u_t|ctx))`。
验证：学过的 LM 困惑度 < 词表大小 K（均匀基线）。

In [ ]:
def compute_perplexity(counts, tokens, n, K, alpha=1.0):
    # TODO: nll=0,cnt=0; 遍历每个预测位置, nll += -log cond_prob(...)[nxt]; 返回 exp(nll/cnt)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
toks = np.array([0,1,2,0,1,2,0,1,2,0,1,2])    # 周期3, 强结构
cnt = count_ngrams(toks, n=2, K=8)
# 用较小的平滑(alpha=0.1)，让强结构序列的困惑度接近确定(≈1)
ppl = compute_perplexity(cnt, toks, n=2, K=8, alpha=0.1)
ppl_unsmoothed_baseline = 8                    # 均匀基线 = K
assert ppl < ppl_unsmoothed_baseline, '强结构序列困惑度应远小于 K=8'
assert ppl < 1.5, '周期3序列 2-gram 几乎确定 -> 困惑度接近 1'
# 大平滑(alpha=1)会抬高困惑度（平滑稀释了确定性）——体会平滑的作用
ppl_a1 = compute_perplexity(cnt, toks, n=2, K=8, alpha=1.0)
assert ppl_a1 > ppl, '更大平滑 -> 更高困惑度'
print(f'周期序列 2-gram 困惑度: alpha=0.1 -> {ppl:.3f}, alpha=1.0 -> {ppl_a1:.3f} (均 < K=8)')
print('✅ 练习 3 通过：困惑度 < 词表大小，证明学到结构（小平滑更逼近确定）')

## ✏️ 练习 4：流式分块无损

实现 `stream_predict(counts, tokens, chunk, n, K, alpha)`：分块预测、维护前 n-1 token 历史，
返回每个位置的预测分布。验证与整段预测逐位相等。

In [ ]:
def stream_predict(counts, tokens, chunk, n, K, alpha=1.0):
    # TODO: 从 pos=n-1 起按 chunk 分块; 每个位置 i 的 ctx=tokens[i-(n-1):i];
    #       append cond_prob(...); 返回 np.array(所有分布)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
toks = np.array([0,1,2,3,2,1,0,1,2,3,2,1,0,1,2,3])
cnt = count_ngrams(toks, n=2, K=8)
def _full(counts, tokens, n, K, alpha=1.0):
    return np.array([cond_prob(counts, tuple(tokens[i-(n-1):i]), K, alpha)
                     for i in range(n-1, len(tokens))])
full = _full(cnt, toks, 2, 8)
for ch in [1, 3, 5, 100]:
    strm = stream_predict(cnt, toks, chunk=ch, n=2, K=8)
    assert strm.shape == full.shape and np.allclose(strm, full), ch
print('✅ 练习 4 通过：任意块大小，流式 == 整段（无损）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def audio_to_tokens(x, codebook, N=256, H=128):
    T = 1 + (len(x) - N)//H
    idx = np.arange(N)[None,:] + H*np.arange(T)[:,None]
    feats = np.abs(np.fft.rfft(x[idx]*hann_window(N)[None,:], axis=1))
    return vq_encode(feats, codebook)

In [ ]:
# 练习 2 参考答案
def count_ngrams(tokens, n, K):
    counts = defaultdict(lambda: np.zeros(K))
    for i in range(len(tokens) - n + 1):
        counts[tuple(tokens[i:i+n-1])][tokens[i+n-1]] += 1
    return counts

def cond_prob(counts, ctx, K, alpha=1.0):
    c = counts.get(ctx, np.zeros(K))
    return (c + alpha) / (c.sum() + alpha*K)

In [ ]:
# 练习 3 参考答案
def compute_perplexity(counts, tokens, n, K, alpha=1.0):
    nll, cnt = 0.0, 0
    for i in range(len(tokens) - n + 1):
        ctx = tuple(tokens[i:i+n-1]); nxt = tokens[i+n-1]
        nll += -np.log(cond_prob(counts, ctx, K, alpha)[nxt]); cnt += 1
    return float(np.exp(nll/cnt))

In [ ]:
# 练习 4 参考答案
def stream_predict(counts, tokens, chunk, n, K, alpha=1.0):
    out, pos = [], n-1
    while pos < len(tokens):
        end = min(pos + chunk, len(tokens))
        for i in range(pos, end):
            out.append(cond_prob(counts, tuple(tokens[i-(n-1):i]), K, alpha))
        pos = end
    return np.array(out)

---
## 🧪 真实数据胶囊：AudioLM / Moshi 的 token 预算

AudioLM 用语义+声学分层、Moshi 用低帧率多流。我们用它们**公开的真实**配置，
算出一段对话音频的 token 序列长度，体会『为什么要低帧率』『多流如何影响序列』。

（纯 numpy/标准库，无需联网、无需任何语音 LM 包。）

In [ ]:
# 真实配置（公开）
CONFIGS = {
    'EnCodec 24kHz (8 流)': dict(frame_rate=75,   n_streams=8),
    'Mimi/Moshi (8 流)':    dict(frame_rate=12.5, n_streams=8),   # Moshi 用 12.5 Hz
    'HuBERT 语义 (1 流)':   dict(frame_rate=50,   n_streams=1),
}

def seq_tokens(audio_sec, frame_rate, n_streams):
    '''一段音频的总 token 数 = 时长 × 帧率 × 流数。'''
    return int(audio_sec * frame_rate * n_streams)

dur = 10  # 10 秒
print(f'{"配置":<24}{"10秒 token 数":>14}')
res = {}
for name, cfg in CONFIGS.items():
    n = seq_tokens(dur, cfg['frame_rate'], cfg['n_streams'])
    res[name] = n
    print(f'{name:<24}{n:>14}')
# Moshi 的低帧率(12.5Hz)让序列比 EnCodec(75Hz)短 6 倍 -> 更利于实时语音 LM
assert res['Mimi/Moshi (8 流)'] < res['EnCodec 24kHz (8 流)']
ratio = res['EnCodec 24kHz (8 流)'] / res['Mimi/Moshi (8 流)']
print(f'\nEnCodec/Mimi 序列长度比 = {ratio:.0f}x（Mimi 低帧率大幅缩短序列）')
assert abs(ratio - 6.0) < 0.1
print('✅ 真实配置：低帧率 token 让语音 LM 的序列更短、更利于实时')

**🧪 胶囊练习**：实现 `context_seconds(max_tokens, frame_rate, n_streams)`：
给定 LM 的最大上下文长度（token 数），算出它能覆盖多少**秒**音频。
对比 EnCodec(75Hz×8) 与 Mimi(12.5Hz×8) 在 8192 token 上下文里能记住多长的对话。

In [ ]:
def context_seconds(max_tokens, frame_rate, n_streams):
    # TODO: max_tokens / (frame_rate * n_streams)
    raise NotImplementedError

In [ ]:
# 自测
enc = context_seconds(8192, 75, 8)
mimi = context_seconds(8192, 12.5, 8)
assert mimi > enc, 'Mimi 低帧率 -> 同样上下文覆盖更长时间'
print(f'8192 token 上下文: EnCodec 覆盖 {enc:.1f}s, Mimi 覆盖 {mimi:.1f}s')
assert abs(mimi - 81.92) < 0.1 and abs(enc - 13.65) < 0.1
print('✅ 胶囊练习通过：低帧率 token 让有限上下文覆盖更长对话')

In [ ]:
# 📖 胶囊参考答案
def context_seconds(max_tokens, frame_rate, n_streams):
    return max_tokens / (frame_rate * n_streams)

### 小结
- 音频 token 化(模块03)让语音可套用文本 LM 全套机器：**音频 token ≈ 文本 token**。
- **语义 token**(内容/紧凑) vs **声学 token**(细节/保真)；AudioLM 分层『语义→声学』。
- **离散语音 LM** = token 上的自回归；**困惑度 < 词表大小 K** 证明学到了结构。
- 文本/音频对齐: 拼接/交错/多流；Moshi 多流实现**全双工**。
- **流式** = 分块边收边算; n-gram 分块**无损等价**整段; 低帧率 token 利于实时。

🎉 **全课闭环**：波形→特征(01)→对齐成文字(02)→压成 token(03)→还原波形(04)→token 语言模型(05)。
你已掌握现代语音系统从信号到大模型的完整骨架。